### README

Identify disease-related variants (pathogenic/likely pathogenic) annottated in ClinVar database located at the +1 position of NAGNAG (2nd N).

*Output coordinates at base 1
**Input data clinvar.vcf.gz is not shared, but can be downloaded from https://www.ncbi.nlm.nih.gov/clinvar/. For this analysis, the data used was donwloaded on July 17th, 2025.

### Requirements

In [1]:
# Install
"""
pandas==2.2.3
"""

'\npandas==2.2.3\n'

In [1]:
# Import libraries
import pandas as pd
import re


### Constants

In [ ]:
# Input: ClinVar data
CLINVAR_PATH = '/PATH/TO/clinvar.vcf.gz'

# Input: annotation data
GTF_PATH = '/PATH/TO/Homo_sapiens.GRCh38.111.gtf.gz'

# Output: set directory
OUTPUT_DIR = '/PATH/TO/OUTPUT'

### Functions

In [3]:
# EXTRACT UPSTREAM AND DOWNSTREAM EXONS COORDINATES FROM GTF FILE

def extract_exon_pairs(gtf_df: pd.DataFrame) -> pd.DataFrame:
   
    # Filter only exon entries
    exon_df = gtf_df[gtf_df['Type'] == 'exon'].copy()

    #replace , by ; in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(',',';')
    #replace : by = in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(':','=')

    # Extract parent mRNA (transcript_id)
    exon_df['mRNA'] = (
        exon_df['Attributes']
        .str.split('transcript_id "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Extract parent gene (gene_id)
    exon_df['Gene'] = (
        exon_df['Attributes']
        .str.split('gene_name "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Sort by gene, transcript, and genomic start
    exon_df = exon_df.sort_values(by=['Gene', 'mRNA', 'Start'])

    # Assign coordinates of the downstream exon ("below")
    exon_df['Start_Below'] = exon_df.groupby('mRNA')['Start'].shift(-1)
    exon_df['End_Below'] = exon_df.groupby('mRNA')['End'].shift(-1)

    # Drop rows where downstream exon doesn't exist
    exon_df = exon_df.dropna(subset=['Start_Below', 'End_Below']).copy()

    # Keep relevant columns and remove duplicates
    exon_df = exon_df[['SeqID', 'Start', 'End', 'Start_Below', 'End_Below', 'Gene', 'STR']]
    exon_df = exon_df.drop_duplicates()
    

    return exon_df

In [4]:
# EXTRACT ALL ANNOTATED NAGNAG FROM GTF FILE

def extract_nagnag(exon_df: pd.DataFrame) -> pd.DataFrame:
    
    df = exon_df.copy()

    df['Acceptor'] = df.apply(lambda row: row['Start_Below'] if row['STR'] == '+' else row['End'], axis=1)

    results = []

    for (chr_, strand_, gene_), group in df.groupby(['SeqID', 'STR', 'Gene']):
        
        group = group.sort_values('Acceptor').reset_index(drop=True)

        for i in range(1, len(group)):
            
            acc1 = group.loc[i-1, 'Acceptor']
            acc2 = group.loc[i, 'Acceptor']
            
            diff = acc2 - acc1
            
            if abs(diff) == 3:
                
                if strand_ == '+':
                    IS_1 = group.loc[i-1, 'End']+1
                    IS_2 = group.loc[i, 'End']+1
                    shortIE = acc1-1
                    longIE = acc2-1

                    results.append({
                        'SeqID': chr_,
                        'Gene': gene_,
                        'STR': strand_,
                        'shortIS': IS_1,
                        'shortIE': shortIE,
                        'longIS': IS_1,
                        'longIE': longIE
                    })

                    if IS_1 != IS_2:
                        results.append({
                            'SeqID': chr_,
                            'Gene': gene_,
                            'STR': strand_,
                            'shortIS': IS_2,
                            'shortIE': shortIE,
                            'longIS': IS_2,
                            'longIE': longIE
                        })


                else:
                    IE_1 = group.loc[i-1, 'Start_Below']-1
                    IE_2 = group.loc[i, 'Start_Below']-1
                    shortIS = acc2+1
                    longIS = acc1+1

                    results.append({
                        'SeqID': chr_,
                        'Gene': gene_,
                        'STR': strand_,
                        'shortIS': shortIS,
                        'shortIE': IE_1,
                        'longIS': longIS,
                        'longIE': IE_1
                    })  

                    if IE_1 != IE_2:
                        results.append({
                            'SeqID': chr_,
                            'Gene': gene_,
                            'STR': strand_,
                            'shortIS': shortIS,
                            'shortIE': IE_2,
                            'longIS': longIS,
                            'longIE': IE_2
                        })

    return pd.DataFrame(results)


In [5]:
# MAP ORIGIN CODES TO MEANINGFUL LABELS

origin_map = {
    '0': 'unknown',
    '1': 'germline',
    '2': 'somatic',
    '4': 'inherited',
    '8': 'paternal',
    '16': 'maternal',
    '32': 'de-novo',
    '64': 'biparental',
    '128': 'uniparental',
    '256': 'not-tested',
    '512': 'tested-inconclusive',
    '1073741824': 'other'
}

def extract_mc(info):
    if pd.isna(info):
        return []
    # Match MC=... until next ; or end of string
    match = re.search(r'MC=([^;]+)(?:;|$)', info)
    if match:
        mc_field = match.group(1)
        entries = mc_field.split(',')
        effects = [entry.split('|')[1] for entry in entries if '|' in entry]
        return effects
    return []

### Analysis

##### 1. Extract coordinates from all annottated NAGNAG 

In [32]:
# genome annotation
gtf_df = pd.read_table(GTF_PATH,
                       names = ['SeqID', 'Source', 'Type', 'Start', 'End', 'Score', 'STR', 'Phase', 'Attributes'],
                       comment='#',
                       low_memory=False)

# get exon pairs
exon_df = extract_exon_pairs(gtf_df)

# get naganag coordinates
nagnag_df = extract_nagnag(exon_df)

# get +1 position at the 3' end
nagnag_df['Position_1_end'] = nagnag_df.apply(lambda row: row['shortIE']+1 if row['STR'] == '+' else row['shortIS']-1, axis=1)

nagnag_df

,SeqID,Gene,STR,shortIS,shortIE,longIS,longIE,Position_1_end
0,1,ADAM15,+,155054507.0,155055786.0,155054507.0,155055789.0,155055787.0
1,1,ADAM15,+,155061194.0,155061414.0,155061194.0,155061417.0,155061415.0
2,1,ADAM15,+,155060833.0,155061414.0,155060833.0,155061417.0,155061415.0
3,1,ARID1A,+,26772633.0,26772811.0,26772633.0,26772814.0,26772812.0
4,1,ATF6,+,161860278.0,161863197.0,161860278.0,161863200.0,161863198.0
...,...,...,...,...,...,...,...,...
2557,Y,SLC25A6,-,1390294.0,1391898.0,1390291.0,1391898.0,1390293.0
2558,Y,TTTY14,-,19045292.0,19048772.0,19045289.0,19048772.0,19045291.0
2559,Y,TTTY14,-,19045292.0,19068723.0,19045289.0,19068723.0,19045291.0
2560,Y,ZBED1,-,2490773.0,2500816.0,2490770.0,2500816.0,2490772.0


##### 2. Select desease related snv variants located at NAGNAG (+1 position)

In [7]:
# read clinvar file
clinvar_df = pd.read_csv(CLINVAR_PATH, sep='\t', comment='#', low_memory=False, header=None, names=['CHROM','POS','ID','REF','ALT','QUAL','FILTER','INFO']) 
# filter only single_nucleotide_variant in info column
clinvar_df = clinvar_df[clinvar_df['INFO'].str.contains('single_nucleotide_variant', na=False)]
clinvar_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
1,1,69134,2205837,A,G,.,.,ALLELEID=2193183;CLNDISDB=MedGen:CN169374;CLND...
2,1,69308,3925305,A,G,.,.,ALLELEID=4039319;CLNDISDB=MedGen:CN169374;CLND...
3,1,69314,3205580,T,G,.,.,ALLELEID=3374047;CLNDISDB=MedGen:CN169374;CLND...
4,1,69404,3925306,T,C,.,.,ALLELEID=4039320;CLNDISDB=MedGen:CN169374;CLND...
5,1,69423,3205581,G,A,.,.,ALLELEID=3374048;CLNDISDB=MedGen:CN169374;CLND...
...,...,...,...,...,...,...,...,...
3660212,NT_187693.1,273806,2219599,G,A,.,.,AF_ESP=0.00055;AF_EXAC=0.00050;ALLELEID=220691...
3660213,NT_187693.1,273866,2237818,A,C,.,.,ALLELEID=2232003;CLNDISDB=MedGen:CN169374;CLND...
3660214,NT_187693.1,274185,3778023,C,T,.,.,ALLELEID=3894028;CLNDISDB=MedGen:C3661900;CLND...
3660215,NT_187693.1,274366,2206666,G,C,.,.,ALLELEID=2200058;CLNDISDB=MedGen:CN169374;CLND...


In [8]:
# merge clinvar_df with nagnag_df on chromosome and position
nagnag_var_df = nagnag_df.merge(clinvar_df, 
                    left_on=['SeqID', 'Position_1_end'], 
                    right_on=['CHROM', 'POS'], 
                    how='inner', 
                    suffixes=('', '_clinvar'))

nagnag_var_df

,SeqID,Gene,STR,shortIS,shortIE,longIS,longIE,Position_1_end,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,1,DHDDS,+,26465826.0,26468891.0,26465826.0,26468894.0,26468892.0,1,26468892,3366967,C,T,.,.,"ALLELEID=3526168;CLNDISDB=MONDO:MONDO:0044326,..."
1,1,DHDDS,+,26460145.0,26468891.0,26460145.0,26468894.0,26468892.0,1,26468892,3366967,C,T,.,.,"ALLELEID=3526168;CLNDISDB=MONDO:MONDO:0044326,..."
2,1,EPB41,+,29039427.0,29053103.0,29039427.0,29053106.0,29053104.0,1,29053104,2493921,C,T,.,.,AF_EXAC=0.00002;ALLELEID=2471639;CLNDISDB=MeSH...
3,1,MSTO1,+,155613734.0,155614058.0,155613734.0,155614061.0,155614059.0,1,155614059,2525306,C,T,.,.,AF_EXAC=0.00014;ALLELEID=2697809;CLNDISDB=MeSH...
4,1,MSTO1,+,155613767.0,155614058.0,155613767.0,155614061.0,155614059.0,1,155614059,2525306,C,T,.,.,AF_EXAC=0.00014;ALLELEID=2697809;CLNDISDB=MeSH...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,8,VPS13B,+,99384318.0,99391556.0,99384318.0,99391559.0,99391557.0,8,99391557,2760550,C,T,.,.,"ALLELEID=2921730;CLNDISDB=MONDO:MONDO:0008999,..."
110,9,FKTN,+,105607952.0,105615277.0,105607952.0,105615280.0,105615278.0,9,105615278,3655277,C,T,.,.,"ALLELEID=3789420;CLNDISDB=MONDO:MONDO:0000171,..."
111,9,TSC1,-,132910693.0,132911001.0,132910690.0,132911001.0,132910692.0,9,132910692,3720835,G,A,.,.,"ALLELEID=3856639;CLNDISDB=MONDO:MONDO:0008612,..."
112,9,TSC1,-,132910693.0,132911001.0,132910690.0,132911001.0,132910692.0,9,132910692,2751813,G,C,.,.,"ALLELEID=2919475;CLNDISDB=MONDO:MONDO:0008612,..."


In [9]:
# extract CLNSIG, ORIGIN and MC from INFO column
nagnag_var_df['CLNSIG'] = nagnag_var_df['INFO'].str.extract(r'CLNSIG=([^;]+)(?:;|$)')
nagnag_var_df['ORIGIN'] = nagnag_var_df['INFO'].str.extract(r'ORIGIN=([^;]+)(?:;|$)')[0].map(origin_map).fillna('nan')
nagnag_var_df['MC'] = nagnag_var_df['INFO'].apply(extract_mc)

# filter only pathogenic and likely pathogenic variants
nagnag_var_df = nagnag_var_df[nagnag_var_df['CLNSIG'].isin(['Pathogenic', 'Likely_pathogenic'])].copy()

nagnag_var_df

,SeqID,Gene,STR,shortIS,shortIE,longIS,longIE,Position_1_end,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,CLNSIG,ORIGIN,MC
34,15,MPI,+,74890090.0,74890552.0,74890090.0,74890555.0,74890553.0,15,74890553,2769293,C,T,.,.,"ALLELEID=2923568;CLNDISDB=MONDO:MONDO:0011257,...",Pathogenic,germline,"[nonsense, 5_prime_UTR_variant, intron_variant]"
38,16,PKD1,-,2097781.0,2097867.0,2097778.0,2097867.0,2097780.0,16,2097780,1179107,G,A,.,.,"ALLELEID=1168702;CLNDISDB=MONDO:MONDO:0008263,...",Pathogenic,germline,"[nonsense, intron_variant]"
48,17,BRCA1,-,43082576.0,43090943.0,43082573.0,43090943.0,43082575.0,17,43082575,55135,G,A,.,.,"ALLELEID=69802;CLNDISDB=MONDO:MONDO:0011450,Me...",Pathogenic,germline,"[nonsense, non-coding_transcript_variant]"
68,2,DIS3L2,+,232300120.0,232329882.0,232300120.0,232329885.0,232329883.0,2,232329883,970918,C,T,.,.,"ALLELEID=952974;CLNDISDB=MONDO:MONDO:0009965,M...",Pathogenic,germline,"[nonsense, non-coding_transcript_variant, intr..."
71,2,SCN1A,-,166045328.0,166046769.0,166045325.0,166046769.0,166045327.0,2,166045327,620092,G,A,.,.,ALLELEID=611541;CLNDISDB=MedGen:CN517202;CLNDN...,Pathogenic,germline,"[nonsense, non-coding_transcript_variant, 5_pr..."
75,21,AIRE,+,44290069.0,44291094.0,44290069.0,44291097.0,44291095.0,21,44291095,983806,A,T,.,.,"ALLELEID=972364;CLNDISDB=MONDO:MONDO:0009411,M...",Likely_pathogenic,unknown,[nonsense]
84,3,SCN5A,-,38579496.0,38580930.0,38579493.0,38580930.0,38579495.0,3,38579495,3699241,G,A,.,.,ALLELEID=3833419;CLNDISDB=MedGen:C3661900;CLND...,Pathogenic,germline,"[nonsense, non-coding_transcript_variant, intr..."
109,8,VPS13B,+,99384318.0,99391556.0,99384318.0,99391559.0,99391557.0,8,99391557,2760550,C,T,.,.,"ALLELEID=2921730;CLNDISDB=MONDO:MONDO:0008999,...",Pathogenic,germline,[nonsense]
110,9,FKTN,+,105607952.0,105615277.0,105607952.0,105615280.0,105615278.0,9,105615278,3655277,C,T,.,.,"ALLELEID=3789420;CLNDISDB=MONDO:MONDO:0000171,...",Pathogenic,germline,"[nonsense, non-coding_transcript_variant]"


##### 3. Filter and save selected variants coordinates dataframe

In [10]:
# change gene and chr col name
nagnag_var_df = nagnag_var_df.rename(columns={'Gene': 'GENE', 'SeqID': 'CHR'})
# keep only relevant columns
nagnag_var_df = nagnag_var_df[['GENE', 'CHR', 'STR', 
         'shortIS', 'shortIE', 'longIS', 'longIE',
         'POS', 'REF', 'ALT', 'CLNSIG', 'ORIGIN', 'MC', 'INFO']].copy()

nagnag_var_df


,GENE,CHR,STR,shortIS,shortIE,longIS,longIE,POS,REF,ALT,CLNSIG,ORIGIN,MC,INFO
34,MPI,15,+,74890090.0,74890552.0,74890090.0,74890555.0,74890553,C,T,Pathogenic,germline,"[nonsense, 5_prime_UTR_variant, intron_variant]","ALLELEID=2923568;CLNDISDB=MONDO:MONDO:0011257,..."
38,PKD1,16,-,2097781.0,2097867.0,2097778.0,2097867.0,2097780,G,A,Pathogenic,germline,"[nonsense, intron_variant]","ALLELEID=1168702;CLNDISDB=MONDO:MONDO:0008263,..."
48,BRCA1,17,-,43082576.0,43090943.0,43082573.0,43090943.0,43082575,G,A,Pathogenic,germline,"[nonsense, non-coding_transcript_variant]","ALLELEID=69802;CLNDISDB=MONDO:MONDO:0011450,Me..."
68,DIS3L2,2,+,232300120.0,232329882.0,232300120.0,232329885.0,232329883,C,T,Pathogenic,germline,"[nonsense, non-coding_transcript_variant, intr...","ALLELEID=952974;CLNDISDB=MONDO:MONDO:0009965,M..."
71,SCN1A,2,-,166045328.0,166046769.0,166045325.0,166046769.0,166045327,G,A,Pathogenic,germline,"[nonsense, non-coding_transcript_variant, 5_pr...",ALLELEID=611541;CLNDISDB=MedGen:CN517202;CLNDN...
75,AIRE,21,+,44290069.0,44291094.0,44290069.0,44291097.0,44291095,A,T,Likely_pathogenic,unknown,[nonsense],"ALLELEID=972364;CLNDISDB=MONDO:MONDO:0009411,M..."
84,SCN5A,3,-,38579496.0,38580930.0,38579493.0,38580930.0,38579495,G,A,Pathogenic,germline,"[nonsense, non-coding_transcript_variant, intr...",ALLELEID=3833419;CLNDISDB=MedGen:C3661900;CLND...
109,VPS13B,8,+,99384318.0,99391556.0,99384318.0,99391559.0,99391557,C,T,Pathogenic,germline,[nonsense],"ALLELEID=2921730;CLNDISDB=MONDO:MONDO:0008999,..."
110,FKTN,9,+,105607952.0,105615277.0,105607952.0,105615280.0,105615278,C,T,Pathogenic,germline,"[nonsense, non-coding_transcript_variant]","ALLELEID=3789420;CLNDISDB=MONDO:MONDO:0000171,..."


In [13]:
# save to csv
nagnag_var_df.to_csv(f'{OUTPUT_DIR}/ClinVar_variants.csv', index=False)